In [2]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords

# ensure stopwords available
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

# -----------------------------------------
# Text preprocessing function
# -----------------------------------------
def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Normalize smart quotes
    text = (text.replace("’", "'")
                .replace("‘", "'")
                .replace("“", '"')
                .replace("”", '"'))

    # Remove punctuation except apostrophes
    for c in string.punctuation:
        if c != "'":
            text = text.replace(c, " ")

    # remove extra spaces
    text = " ".join(text.split())

    # remove stopwords
    tokens = text.split()
    clean = [w for w in tokens if w not in STOPWORDS]
    return " ".join(clean)


# -----------------------------------------
# Main modular loader
# -----------------------------------------
def load_and_preprocess(fake_path, true_path):
    # Load datasets
    fake = pd.read_csv(fake_path)
    true = pd.read_csv(true_path)

    # add labels
    fake["label"] = 0
    true["label"] = 1

    # combine
    data = pd.concat([fake, true]).reset_index(drop=True)

    # only keep relevant columns
    data = data[["title", "text", "label"]]

    # combine title+text
    data["combined"] = data["title"].astype(str) + " " + data["text"].astype(str)

    # preprocess
    data["preprocessed"] = data["combined"].apply(preprocess_text)

    return data

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

# ============================
# Data
# ============================
data = load_and_preprocess("/content/fake.csv", "/content/true.csv")
X = data['preprocessed']   # text (Series)
y = data['label']          # 0/1 labels

# ============================
# Build ensemble model: LR + SVM + Tree with Voting
# ============================
log_reg = LogisticRegression(max_iter=500, random_state=42)
svm     = LinearSVC(random_state=42)
tree    = DecisionTreeClassifier(max_depth=20, random_state=42)

voting_clf = VotingClassifier(
    estimators=[
        ('lr', log_reg),
        ('svm', svm),
        ('tree', tree),
    ],
    voting='hard'  # majority vote
)

# Pipeline: TF-IDF -> Voting Ensemble
def make_model():
    return Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50000)), # vectorize X data
        ('clf', voting_clf)
    ])

# ============================
# 5-fold stratified CV
# ============================
kfolds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []

for fold_idx, (train_idx, test_idx) in enumerate(kfolds.split(X, y), start=1):
    X_train_text, X_test_text = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = make_model()              # new pipeline for this fold
    model.fit(X_train_text, y_train)  # TF-IDF + ensemble trained here

    y_pred = model.predict(X_test_text)
    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)

    print(f"Fold {fold_idx}")
    print("  Train size:", len(X_train_text))
    print("  Test size:", len(X_test_text))
    print("  Fake % in Train:", y_train.mean())
    print("  Fake % in Test:", y_test.mean())
    print(f"  Accuracy: {acc:.4f}\n")

# ============================
# Mean CV score across folds
# ============================
mean_acc = np.mean(fold_accuracies)
print("=====================================")
print(f"Mean 5-fold CV Accuracy: {mean_acc:.4f}")
print(f"Mean 5-fold CV Error:    {1 - mean_acc:.4f}")
print("=====================================")

# ============================
# Train final ensemble on full data
# ============================
final_model = make_model()
final_model.fit(X, y)

Fold 1
  Train size: 35918
  Test size: 8980
  Fake % in Train: 0.47703101508992707
  Fake % in Test: 0.4769487750556793
  Accuracy: 0.9954

Fold 2
  Train size: 35918
  Test size: 8980
  Fake % in Train: 0.47700317389609664
  Fake % in Test: 0.47706013363028954
  Accuracy: 0.9953

Fold 3
  Train size: 35918
  Test size: 8980
  Fake % in Train: 0.47700317389609664
  Fake % in Test: 0.47706013363028954
  Accuracy: 0.9961

Fold 4
  Train size: 35919
  Test size: 8979
  Fake % in Train: 0.47701773434672456
  Fake % in Test: 0.4770018933066043
  Accuracy: 0.9965

Fold 5
  Train size: 35919
  Test size: 8979
  Fake % in Train: 0.47701773434672456
  Fake % in Test: 0.4770018933066043
  Accuracy: 0.9967

Mean 5-fold CV Accuracy: 0.9960
Mean 5-fold CV Error:    0.0040


Pipeline(steps=[('tfidf', TfidfVectorizer(max_features=50000)),
                ('clf',
                 VotingClassifier(estimators=[('lr',
                                               LogisticRegression(max_iter=500,
                                                                  random_state=42)),
                                              ('svm',
                                               LinearSVC(random_state=42)),
                                              ('tree',
                                               DecisionTreeClassifier(max_depth=20,
                                                                      random_state=42))]))])

Results with logistics regression only:

Fold 1
  Train size: 35918
  Test size: 8980
  Fake % in Train: 0.47703101508992707
  Fake % in Test: 0.4769487750556793
  Accuracy: 0.9850

Fold 2
  Train size: 35918
  Test size: 8980
  Fake % in Train: 0.47700317389609664
  Fake % in Test: 0.47706013363028954
  Accuracy: 0.9878

Fold 3
  Train size: 35918
  Test size: 8980
  Fake % in Train: 0.47700317389609664
  Fake % in Test: 0.47706013363028954
  Accuracy: 0.9896

Fold 4
  Train size: 35919
  Test size: 8979
  Fake % in Train: 0.47701773434672456
  Fake % in Test: 0.4770018933066043
  Accuracy: 0.9899

Fold 5
  Train size: 35919
  Test size: 8979
  Fake % in Train: 0.47701773434672456
  Fake % in Test: 0.4770018933066043
  Accuracy: 0.9893

=====================================
Mean 5-fold CV Accuracy: 0.9883
Mean 5-fold CV Error:    0.0117
=====================================

In [ ]:
# ============================================
# Evaluation block: generalization error + ROC/PR curves
# ============================================
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.model_selection import cross_val_predict
from sklearn.svm import SVC

# Generalization error
gen_error = 1 - mean_acc
print("=====================================")
print(f"Estimated generalization error (1 - mean CV accuracy): {gen_error:.4f}")
print("=====================================")

# 2. Build a probabilistic ensemble (soft voting) for ROC/PR evaluation
#    Similar to your original ensemble, but:
#    - use SVC(probability=True) instead of LinearSVC
#    - use voting='soft' so we get predict_proba

log_reg_soft = LogisticRegression(max_iter=500, random_state=42)
svm_soft     = SVC(kernel='linear', probability=True, random_state=42)
tree_soft    = DecisionTreeClassifier(max_depth=20, random_state=42)

soft_voting_clf = VotingClassifier(
    estimators=[
        ('lr',   log_reg_soft),
        ('svm',  svm_soft),
        ('tree', tree_soft),
    ],
    voting='soft'  # now we can use predict_proba
)

soft_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000)),
    ('clf',   soft_voting_clf)
])

# 3. Get cross-validated predicted probabilities for the positive class (label=1)
#    We use cross_val_predict to avoid optimistic bias.

# load test data set
evaluation_data = load_and_preprocess("/content/Fake.csv", "/content/True.csv")
X = evaluation_data['preprocessed']   # text (Series)
y = evaluation_data['label']          # 0/1 labels

y_scores = cross_val_predict(
    soft_model,
    X,
    y,
    cv=5,
    method="predict_proba"
)[:, 1]

# 4. ROC curve + AUC
fpr, tpr, _ = roc_curve(y, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (5-fold CV, soft voting ensemble)")
plt.legend()
plt.grid(True)
plt.show()

# 5. Precision-Recall curve + AUPRC
precision, recall, _ = precision_recall_curve(y, y_scores)
pr_auc = average_precision_score(y, y_scores)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"PR curve (AUPRC = {pr_auc:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (5-fold CV, soft voting ensemble)")
plt.legend()
plt.grid(True)
plt.show()

print("=====================================")
print(f"ROC AUC:   {roc_auc:.4f}")
print(f"PR AUC:    {pr_auc:.4f}  (AUPRC)")
print("=====================================")

Estimated generalization error (1 - mean CV accuracy): 0.0040


In [ ]:
#https://www.politico.com/news/2025/11/18/trump-admin-deportation-order-00657401
real = "The Trump administration has acknowledged that officials improperly deported a transgender woman to Mexico last week in violation of an immigration judge’s order in March concluding she was likely to face torture in her home country. Now, the administration is working to bring Britania Uriostegui Rios back to the U.S. from Tijuana — perhaps as soon as Tuesday afternoon — while it attempts to find an alternative country for her deportation. Uriostegui Rios is suing to force the administration to release her from custody when she returns to the U.S. Uriostegui Rios, a Nevada resident with a long criminal rap sheet, lost her status as a lawful permanent resident in 2023, after pleading guilty to a felony assault with a deadly weapon. After she received a suspended criminal sentence, Uriostegui was quickly placed in deportation proceedings, and an immigration judge ordered her deported from the country earlier this year. However, the judge also barred the administration from sending Uriostegui Rios to Mexico, finding a likelihood she would be tortured or killed as a result of her transgender status. Despite that order, lawyers for Uriostegui Rios say that on Nov. 11, without warning, she was abruptly transported from Louisiana to Texas and placed on a bus that dropped her off in Mexico. After the attorneys inquired about the deportation, the Justice Department acknowledged the error. “ICE confirmed that your client was removed to Mexico inadvertently,” a DOJ attorney wrote in a Nov. 12 email filed in federal court. The attorney added a day later: “ICE stands ready to remedy the inadvertent removal by allowing your client to voluntarily reenter the United States if your client wishes to do so.” The Department of Homeland Security did not immediately respond to a request for comment. It’s the latest in a string of errors that have at times marred the Trump administration’s mass deportation agenda.The Trump administration sparked national headlines and an ongoing legal battle when it illegally deported a Salvadoran man, Kilmar Abrego Garcia, to his home country despite an immigration judge’s ruling that not be sent there because he could be targeted for gang violence. The administration also worked to facilitate the return of a Guatemalan man who was deported to Mexico without being afforded a chance to lodge fear of persecution there.Uriostegui Rios’ lawyers say the erroneous deportation was likely only discovered because she was one of the few deportees fortunate enough to have lawyers.“The administration says that the immigration courts ultimately are the final arbiters on these issues and decisions. … Yet they still flagrantly violated it,” said Nora Ahmed, legal director for the ACLU of Louisiana, which is representing Uriostegui Rios. “There can be no excuse for that. It’s not an ‘oops.’ How can you ‘oops’ if someone dies?”Uriostegui Rios’ lawyers say their client should be released in part to avoid leaving her in the hands of the same agency that erroneously deported her in the first place. They also say Uriostegui Rios has already spent months in detention without a successful effort to deport her to a safe foreign country, testing the constitutional limits for holding someone while they await deportation. Since March, the administration has attempted, so far without success, to send Uriostegui Rios to Costa Rica, Nicaragua, Honduras and El Salvador."
real = real[:1000]

#https://www.reuters.com/world/us/trump-approval-falls-lowest-his-term-over-prices-epstein-files-reutersipsos-poll-2025-11-18/
reuter = "WASHINGTON, Nov 18 (Reuters) - President Donald Trump's approval rating fell to 38%, the lowest since his return to power, with Americans unhappy about his handling of the high cost of living and the investigation into the late convicted sex offender Jeffrey Epstein, a Reuters/Ipsos poll found.The four-day poll, which concluded on Monday, comes as Trump's grip on his Republican Party shows signs of weakening. The Republican-controlled House of Representatives on Tuesday passed a measure to force the release of Justice Department files on Epstein. Trump had opposed the move for months while one of his closest supporters in Congress, Representative Marjorie Taylor Greene, turned into a harsh critic over his resistance. Trump reversed his position on Sunday as lawmakers prepared to move forward without him. The survey showed Trump's overall approval has fallen two percentage points since a Reuters/Ipsos poll in early November. The poll, which was conducted online, surveyed 1,017 U.S. adults nationwide and had a margin of error of about 3 percentage points. Trump started his second term in office with 47% of Americans giving him a thumbs up. The nine-point decline since January leaves his overall popularity near the lows seen during his first term in office, and close to the weakest ratings for his Democratic predecessor in the White House, Joe Biden. Biden's approval rating sank as low as 35% while Trump's first-term popularity fell as low as 33%. Trump started his second term in office with 47% of Americans giving him a thumbs up. The nine-point decline since January leaves his overall popularity near the lows seen during his first term in office, and close to the weakest ratings for his Democratic predecessor in the White House, Joe Biden. Biden's approval rating sank as low as 35% while Trump's first-term popularity fell as low as 33%. Trump's signature economic policy push has been to hike taxes on imported goods to prop up American manufacturing, but many economists believe the policy has led to higher prices. Expressing frustration over the public perception of his handling of the economy, Trump last week dialed back import taxes on coffee, beef, bananas and other staples. His sagging popularity could make Republicans more vulnerable in next year's congressional elections, though the Reuters/Ipsos poll showed voters continue to see Trump's Republican Party as having a better approach to economic policy.What we're seeing is probably the biggest test of his presidency in terms of his grip on the Republican Party, said Mike Ongstad, an independent strategist and former Republican who has not supported Trumps presidential campaigns.Only 20% of Americans - including just 44% of Republicans - approve of how Trump has handled the Epstein case, the Reuters/Ipsos poll showed. Some 70% of poll respondents - including 87% of Democrats and 60% of Republicans - said they believe the government is hiding information about Epsteins clients."
reuter = reuter[:1000]

# note: shortened to 1000 characters as training data are around ~1000 characters

real = preprocess_text(real)
reuter = preprocess_text(reuter)

texts = [real, reuter]
names = ["Politico article", "Reuters article"]

# predict
preds = final_model.predict(texts)

label_map = {0: "FAKE", 1: "REAL"}  # adjust if your labels are flipped

for name, pred in zip(names, preds):
    print(f"{name}: {label_map[pred]}")


Politico article: FAKE
Reuters article: REAL
